In [73]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import json
from pathlib import Path

class TargetLabelEncoder:
    def __init__(self):
        self._le = LabelEncoder()
        self.classes_ = None

    def fit(self, y):
        y = pd.Series(y) if not isinstance(y, (pd.Series, pd.Index)) else y
        self._le.fit(y.astype(str) if y.dtype == "object" else y)
        self.classes_ = list(self._le.classes_)
        return self

    def transform(self, y):
        y = pd.Series(y) if not isinstance(y, (pd.Series, pd.Index)) else y
        return self._le.transform(y.astype(str) if y.dtype == "object" else y)

    def fit_transform(self, y):
        return self.fit(y).transform(y)

    def inverse_transform(self, y_encoded):
        return self._le.inverse_transform(np.asarray(y_encoded))

def encode_labels(y, encoder: TargetLabelEncoder | None = None):
    if encoder is None:
        enc = TargetLabelEncoder().fit(y)
        return enc.transform(y), enc
    else:
        return encoder.transform(y), encoder

def split_xy(df, target):
    y = df[target]
    X = df.drop(columns=[target])
    return X, y

class MissingValueHandler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = X.copy()
        self.int_cols_ = X.select_dtypes(include=[np.integer]).columns.tolist()
        self.float_cols_ = X.select_dtypes(include=[np.floating]).columns.tolist()
        self.cat_cols_ = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
        self.medians_ = X[self.int_cols_].median(numeric_only=True)
        self.means_ = X[self.float_cols_].mean(numeric_only=True)
        return self
    def transform(self, X):
        X = X.copy()
        if self.int_cols_:   
            X[self.int_cols_] = X[self.int_cols_].fillna(self.medians_)
        if self.float_cols_: 
            X[self.float_cols_] = X[self.float_cols_].fillna(self.means_)
        if self.cat_cols_:   
            X[self.cat_cols_] = X[self.cat_cols_].fillna("(NA)").astype("string")
        return X

def build_two_stage_preprocessor():
    stage1 = MissingValueHandler()
    enc_scale = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), make_column_selector(dtype_include=[np.number])),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                    make_column_selector(dtype_include=["object","string","category"])),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return Pipeline([("stage1_missing", stage1), ("stage2_encode_scale", enc_scale)])

def make_folds(y, n_splits=5, seed=42, out=None, name="dataset"):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = [{"train_idx": tr.tolist(), "valid_idx": va.tolist()} for tr,va in skf.split(np.zeros(len(y)), y)]
    if out:
        Path(out).mkdir(parents=True, exist_ok=True)
        Path(out, f"{name}_skf{n_splits}.json").write_text(json.dumps(folds))
    return folds



In [74]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/raw")
AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "aps": {"file": "aps.csv", "target": "class", "na_values": ["na"]},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"},
    "jannis": {"file": "jannis.csv", "target": "__target__"},
}

def _exists(name): 
    return (DATA_DIR / AVAILABLE[name]["file"]).exists()

print("Available files:")
for k in AVAILABLE: 
    print(f" - {k:10s} -> {AVAILABLE[k]['file']}  {'✓' if _exists(k) else '✗'}")

Available files:
 - car        -> car.csv  ✓
 - aps        -> aps.csv  ✓
 - covertype  -> covertype.csv  ✓
 - jannis     -> jannis.csv  ✓


In [75]:
DATASET = "jannis"

In [76]:
from IPython.display import display
import pandas as pd

name = DATASET
info = AVAILABLE[name]
path = (DATA_DIR / info["file"])
read_kwargs = {}
print(info)
if "na_values" in info: 
    read_kwargs["na_values"] = info["na_values"]

df = pd.read_csv(path, **read_kwargs)
print(f"Loaded: {name} -> {path.name}")
print("Shape:", df.shape)
print("Memory MB (approx):", round(df.memory_usage(index=True, deep=True).sum()/1e6, 2))
print("Columns sample:", list(df.columns)[:12], "..." if df.shape[1] > 12 else "")
display(df.head())

{'file': 'jannis.csv', 'target': '__target__'}
Loaded: jannis -> jannis.csv
Shape: (83733, 55)
Memory MB (approx): 36.84
Columns sample: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12'] ...


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V46,V47,V48,V49,V50,V51,V52,V53,V54,__target__
0,0.047095,0.316667,0.288889,0.180925,0.101677,0.252014,0.235673,0.054190,0.485197,253.6360,...,0.421830,0.401547,0.036195,153.2040,-7.584500,21.9080,13.0200,0.086289,1.074020,3
1,0.464873,1.000000,0.483333,0.502843,0.233860,0.998355,0.465075,0.016096,0.038194,141.3640,...,0.356803,0.439840,0.022729,123.5940,0.813364,19.8483,18.3443,0.039786,0.333176,3
2,0.053872,0.516667,0.180556,0.431467,0.064608,0.412341,0.151937,0.056719,0.422519,238.9570,...,0.583339,0.611528,0.042459,147.7000,-0.428918,35.7166,15.8205,0.164682,1.175180,3
3,0.030475,0.245833,0.175000,0.128515,0.438525,0.207337,0.146200,0.049183,0.291633,52.4554,...,0.553694,0.417488,0.059169,147.5880,0.108469,13.0906,21.0650,0.028791,1.215070,1
4,0.038883,0.256250,0.225000,0.128165,0.618360,0.252958,0.171803,0.059235,0.325605,60.1405,...,0.453029,0.687470,0.051630,85.8803,-1.058330,23.8815,10.5757,0.005814,0.222675,1


In [77]:
from sklearn.model_selection import train_test_split

target_col = info["target"]

X, y = split_xy(df, target_col)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"X_train size: {X_train.shape}, X_test size: {X_test.shape}")
print(f"y_train size: {y_train.shape}, y_test size: {y_test.shape}")

X_train size: (66986, 54), X_test size: (16747, 54)
y_train size: (66986,), y_test size: (16747,)


Train will be used for hyper-parameter search among 2 methods (Random and Bayes). k-fold cross validation will be performed on this subset of the original dataset.  
Test will be used for final validation and training the final models that will be compared in order to establish new the best hyper-parameter combination. 
This approach prevents hyper-parameters from fitting to the data that would be later used for validation (leakage).     

In [78]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

label_enc = TargetLabelEncoder().fit(y_train)
y_train_enc = label_enc.transform(y_train)
y_test_enc  = label_enc.transform(y_test)

preprocessor = build_two_stage_preprocessor()

xgb_base = XGBClassifier(
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    eval_metric="logloss",
    n_estimators=5000,
    random_state=42,
)

pipe = Pipeline([
    ("prep", preprocessor),
    ("xgb",  xgb_base),
])

print("Pipeline ready. Label encoder fitted on y_train.")

Pipeline ready. Label encoder fitted on y_train.


In [43]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import make_scorer, accuracy_score
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from scipy.stats import randint, uniform, loguniform

def run_random_search(
    param_distributions,
    X, y,
    n_iter=50,
    cv_splits=10,
    random_state=42,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1,
):

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    scorer = make_scorer(accuracy_score) if scoring == "accuracy" else scoring

    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
    )
    search.fit(X, y)

    results_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values("mean_test_score", ascending=False)
        .reset_index(drop=True)
    )

    best_params_plain = {k.replace("xgb__", ""): v for k, v in search.best_params_.items()}
    best_score = float(search.best_score_)
    return best_params_plain, best_score, results_df, search

In [ ]:
# small test run
random_grid = {
    "xgb__n_estimators": randint(100, 201),
    "xgb__max_depth": randint(4, 6),
    "xgb__learning_rate": loguniform(1e-2, 1e-1),
    "xgb__booster": ["gbtree"],
}

In [46]:
best_params_r, best_score_r, results_r, rs_obj = run_random_search(
    param_distributions=random_grid,
    X=X_train,                      
    y=y_train_enc,
    n_iter=5, cv_splits=2, random_state=42, n_jobs=-1, verbose=1
)
print(best_score_r, best_params_r)

Fitting 2 folds for each of 5 candidates, totalling 10 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [22:57:52] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [22:57:52] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [22:57:53] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

0.6889499298360852 {'booster': 'gbtree', 'learning_rate': np.float64(0.05395030966670229), 'max_depth': 4, 'n_estimators': 120}


In [48]:
random_grid = {
    "xgb__n_estimators": randint(100, 4001),
    "xgb__max_depth": randint(2, 13),
    "xgb__learning_rate": loguniform(1e-2, 3e-1),
    "xgb__subsample": uniform(0.5, 0.5),          # 0.5–1.0
    "xgb__colsample_bytree": uniform(0.5, 0.5),   # 0.5–1.0
    "xgb__min_child_weight": loguniform(1e-2, 1e2),
    "xgb__reg_lambda": loguniform(1e-3, 1e2),
    "xgb__reg_alpha": loguniform(1e-3, 1e2),
    "xgb__gamma": uniform(0.0, 10.0),
    "xgb__max_bin": randint(128, 1025),
    "xgb__booster": ["gbtree"],
}

In [49]:
best_params_r, best_score_r, results_r, rs_obj = run_random_search(
    param_distributions=random_grid,
    X=X_train,                      
    y=y_train_enc,
    n_iter=50, cv_splits=4, random_state=42, n_jobs=-1, verbose=1
)
print(best_score_r, best_params_r)

Fitting 4 folds for each of 50 candidates, totalling 200 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:02:00] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:02:00] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:02:00] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

0.7230614890901598 {'booster': 'gbtree', 'colsample_bytree': np.float64(0.6254302636733307), 'gamma': np.float64(1.8433367433137005), 'learning_rate': np.float64(0.013166161498490977), 'max_bin': 254, 'max_depth': 10, 'min_child_weight': np.float64(5.675440866816694), 'n_estimators': 1764, 'reg_alpha': np.float64(0.007353119099448618), 'reg_lambda': np.float64(0.006055990892372906), 'subsample': np.float64(0.6251214490822976)}


In [50]:
print(best_score_r, best_params_r)

0.7230614890901598 {'booster': 'gbtree', 'colsample_bytree': np.float64(0.6254302636733307), 'gamma': np.float64(1.8433367433137005), 'learning_rate': np.float64(0.013166161498490977), 'max_bin': 254, 'max_depth': 10, 'min_child_weight': np.float64(5.675440866816694), 'n_estimators': 1764, 'reg_alpha': np.float64(0.007353119099448618), 'reg_lambda': np.float64(0.006055990892372906), 'subsample': np.float64(0.6251214490822976)}


In [67]:
random_grid = {
    "xgb__n_estimators": randint(100, 2001),
    "xgb__max_depth": randint(2, 13),
    "xgb__learning_rate": loguniform(1e-2, 3e-1),
    "xgb__subsample": uniform(0.5, 0.5),
    "xgb__colsample_bytree": uniform(0.5, 0.5),
    "xgb__min_child_weight": loguniform(1e-2, 1e2),
    "xgb__reg_lambda": loguniform(1e-3, 1e2),
    "xgb__reg_alpha": loguniform(1e-3, 1e2),
    "xgb__gamma": uniform(0.0, 10.0),
    "xgb__max_bin": randint(128, 1025),
    "xgb__booster": ["gbtree"],
}

In [68]:
AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "aps": {"file": "aps.csv", "target": "class", "na_values": ["na"]},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"}
}

In [69]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

DATA_DIR = Path("data/raw")
N_ITER = 50
CV_SPLITS = 4
RANDOM_STATE = 42
TEST_SIZE = 0.2

results_summary = {}

for name, info in AVAILABLE.items():
    print(f"\n=== Dataset: {name} ===")
    file_path = DATA_DIR / info["file"]

    # 1) Read
    df = pd.read_csv(file_path, na_values=info.get("na_values"))

    # 2) __STRICT split__: one held-out test per dataset
    X, y = split_xy(df, info["target"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    # 3) Label encoding on TRAIN only
    le = TargetLabelEncoder().fit(y_train)
    y_train_enc = le.transform(y_train)
    y_test_enc  = le.transform(y_test)

    # 4) Fresh pipeline per dataset
    preprocessor = build_two_stage_preprocessor()
    xgb = XGBClassifier(
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )
    pipe = Pipeline([("prep", preprocessor), ("xgb", xgb)])  # `pipe` must be global for run_random_search
    globals()["pipe"] = pipe  # ensure run_random_search sees this pipeline

    # 5) Randomized search on RAW X_train (pipeline handles preprocessing inside CV)
    best_params_r, best_score_r, results_r, rs_obj = run_random_search(
        param_distributions=random_grid,
        X=X_train,
        y=y_train_enc,
        n_iter=N_ITER,
        cv_splits=CV_SPLITS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
    )

    # 6) Store clean results (cast numpy types to native Python)
    def _to_native(d):
        out = {}
        for k, v in d.items():
            if hasattr(v, "item"):
                out[k] = v.item()
            else:
                out[k] = v
        return out

    results_summary[name] = {
        "cv_best_accuracy": float(best_score_r),
        "best_params": _to_native(best_params_r),
    }

    print(f"Best CV accuracy ({name}): {best_score_r:.4f}")
    print("Best params:", results_summary[name]["best_params"])

# After loop: `results_summary` holds the best config per dataset
print("\n=== Summary (RandomizedSearchCV) ===")
for ds, res in results_summary.items():
    print(f"{ds}: acc={res['cv_best_accuracy']:.4f} | params={res['best_params']}")



=== Dataset: car ===
Fitting 4 folds for each of 50 candidates, totalling 200 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:38:02] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:38:02] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:38:02] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fo

Best CV accuracy (car): 0.9689
Best params: {'booster': 'gbtree', 'colsample_bytree': 0.5979914312095727, 'gamma': 0.45227288910538066, 'learning_rate': 0.03023795012558475, 'max_bin': 847, 'max_depth': 3, 'min_child_weight': 1.4413469371110335, 'n_estimators': 891, 'reg_alpha': 0.06078083099681954, 'reg_lambda': 0.02539057572102413, 'subsample': 0.7713480415791243}

=== Dataset: aps ===
Fitting 4 folds for each of 50 candidates, totalling 200 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:47:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:47:06] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:47:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

Best CV accuracy (aps): 0.9943
Best params: {'booster': 'gbtree', 'colsample_bytree': 0.7838501639099957, 'gamma': 0.31313292455558583, 'learning_rate': 0.1754513616606681, 'max_bin': 683, 'max_depth': 3, 'min_child_weight': 0.3807158379249394, 'n_estimators': 1095, 'reg_alpha': 0.9761125443110458, 'reg_lambda': 40.679084943595456, 'subsample': 0.5442462510259598}

=== Dataset: covertype ===
Fitting 4 folds for each of 50 candidates, totalling 200 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:59:31] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:59:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [00:59:32] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

Best CV accuracy (covertype): 0.9579
Best params: {'booster': 'gbtree', 'colsample_bytree': 0.9224376554847273, 'gamma': 0.23271935735825866, 'learning_rate': 0.15961316409649484, 'max_bin': 744, 'max_depth': 8, 'min_child_weight': 0.02969335773129637, 'n_estimators': 1624, 'reg_alpha': 0.055167464252320046, 'reg_lambda': 48.221504848249324, 'subsample': 0.5195931633378231}

=== Summary (RandomizedSearchCV) ===
car: acc=0.9689 | params={'booster': 'gbtree', 'colsample_bytree': 0.5979914312095727, 'gamma': 0.45227288910538066, 'learning_rate': 0.03023795012558475, 'max_bin': 847, 'max_depth': 3, 'min_child_weight': 1.4413469371110335, 'n_estimators': 891, 'reg_alpha': 0.06078083099681954, 'reg_lambda': 0.02539057572102413, 'subsample': 0.7713480415791243}
aps: acc=0.9943 | params={'booster': 'gbtree', 'colsample_bytree': 0.7838501639099957, 'gamma': 0.31313292455558583, 'learning_rate': 0.1754513616606681, 'max_bin': 683, 'max_depth': 3, 'min_child_weight': 0.3807158379249394, 'n_estima

In [79]:
def run_bayes_search(
    search_spaces,         # dict of skopt spaces, keys like "xgb__max_depth"
    X, y,
    n_iter=50,
    cv_splits=10,
    random_state=42,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1,
    acq_func="EI",
):
    """BayesSearchCV over the Pipeline `pipe` (no per-fold early stopping)."""
    assert 'pipe' in globals(), "Build the pipeline in Cell 3 first."
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    scorer = make_scorer(accuracy_score) if scoring == "accuracy" else scoring

    search = BayesSearchCV(
        estimator=pipe,
        search_spaces=search_spaces,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        refit=True,
        optimizer_kwargs={"acq_func": acq_func},
    )
    search.fit(X, y)

    results_df = (
        pd.DataFrame(search.cv_results_)
        .sort_values("mean_test_score", ascending=False)
        .reset_index(drop=True)
    )
    best_params_plain = {k.replace("xgb__", ""): v for k, v in search.best_params_.items()}
    best_score = float(search.best_score_)
    return best_params_plain, best_score, results_df, search


In [ ]:
bayes_spaces = {
    "xgb__n_estimators": Integer(400, 2000),
    "xgb__max_depth": Integer(2, 12),
    "xgb__learning_rate": Real(1e-2, 3e-1, prior="log-uniform"),
    "xgb__subsample": Real(0.5, 1.0, prior="uniform"),
    "xgb__colsample_bytree": Real(0.5, 1.0, prior="uniform"),
    "xgb__min_child_weight": Real(1e-2, 1e2, prior="log-uniform"),
    "xgb__reg_lambda": Real(1e-3, 1e2, prior="log-uniform"),
    "xgb__reg_alpha": Real(1e-3, 1e2, prior="log-uniform"),
    "xgb__gamma": Real(0.0, 10.0, prior="uniform"),
    "xgb__max_bin": Integer(128, 1024),
    "xgb__booster": Categorical(["gbtree"]),
}

AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"}
}

In [83]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

DATA_DIR = Path("data/raw")
N_ITER = 50
CV_SPLITS = 4
RANDOM_STATE = 42
TEST_SIZE = 0.2

bayes_results_summary = {}

for name, info in AVAILABLE.items():
    print(f"\n=== Dataset: {name} ===")
    file_path = DATA_DIR / info["file"]

    # 1) Read
    df = pd.read_csv(file_path, na_values=info.get("na_values"))

    # 2) __STRICT split__: one held-out test per dataset
    X, y = split_xy(df, info["target"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    # 3) Label encoding on TRAIN only
    le = TargetLabelEncoder().fit(y_train)
    y_train_enc = le.transform(y_train)
    y_test_enc  = le.transform(y_test)

    # 4) Fresh pipeline per dataset
    preprocessor = build_two_stage_preprocessor()
    xgb = XGBClassifier(
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )
    pipe = Pipeline([("prep", preprocessor), ("xgb", xgb)])  # global for run_bayes_search
    globals()["pipe"] = pipe

    # 5) Bayes search on RAW X_train (pipeline handles preprocessing inside CV)
    best_params_b, best_score_b, results_b, bs_obj = run_bayes_search(
        search_spaces=bayes_spaces,
        X=X_train,
        y=y_train_enc,
        n_iter=N_ITER,
        cv_splits=CV_SPLITS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        acq_func="EI",
    )

    # 6) Store results (cast numpy types to native Python for cleanliness)
    def _to_native(d):
        out = {}
        for k, v in d.items():
            try:
                out[k] = v.item()
            except Exception:
                out[k] = v
        return out

    bayes_results_summary[name] = {
        "cv_best_accuracy": float(best_score_b),
        "best_params": _to_native(best_params_b),
    }

    print(f"Best CV accuracy ({name}): {best_score_b:.4f}")
    print("Best params:", bayes_results_summary[name]["best_params"])

print("\n=== Summary (BayesSearchCV) ===")
for ds, res in bayes_results_summary.items():
    print(f"{ds}: acc={res['cv_best_accuracy']:.4f} | params={res['best_params']}")



=== Dataset: car ===
Fitting 4 folds for each of 1 candidates, totalling 4 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:33] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:33] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:33] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU trainin

Fitting 4 folds for each of 1 candidates, totalling 4 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:49] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:49] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [07:17:49] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

Fitting 4 folds for each of 1 candidates, totalling 4 fits


...

Fitting 4 folds for each of 1 candidates, totalling 4 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:19:34] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:19:34] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:19:34] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

Fitting 4 folds for each of 1 candidates, totalling 4 fits


/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:21:45] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:21:45] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/stano/projects/autoML/project/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [08:21:45] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fo

TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [ ]:
AVAILABLE = {
    "car": {"file": "car.csv", "target": "class"},
    "aps": {"file": "aps.csv", "target": "class", "na_values": ["na"]},
    "covertype": {"file": "covertype.csv", "target": "Cover_Type"},
    "jannis": {"file": "jannis.csv", "target": "__target__"},
}